# Task 1

In [3]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import heapq
from abc import ABC, abstractmethod

## Task 1.1 – Discretización del Mundo

In [4]:
# La imagen tiene miles de píxeles. En lugar de tratar cada píxel como un nodo (ineficiente), la dividimos en "tiles" de `tile_size x tile_size` píxeles.
# Cada tile se convierte en 
# un nodo del grafo.

FREE  = 0   # blanco (libre)
WALL  = 1   # negro (pared)
START = 2   # rojo (inicio)
GOAL  = 3   # verde (metas)

def classify_tile(tile_pixels: np.ndarray) -> int:
    # Promedio de R, G, B en todos los píxeles del tile
    avg = tile_pixels.mean(axis=(0, 1))  # shape: (3,)
    r, g, b = avg[0], avg[1], avg[2]
    
    # Clasificacion de color
    if r < 50 and g < 50 and b < 50:
        return WALL
    elif r > 150 and g < 80 and b < 80:
        return START
    elif g > 150 and r < 80 and b < 80:
        return GOAL
    else:
        return FREE


def discretize_image(image_path: str, tile_size: int = 10):
    # 1. Cargar imagen y convertir a RGB
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)  # shape: (alto, ancho, 3)
    H, W, _ = img_array.shape
    print(f"Imagen cargada: {W}x{H} píxeles")
    
    # 2. Calcular dimensiones de la grilla
    rows = H // tile_size
    cols = W // tile_size
    print(f"Grilla discreta: {rows}x{cols} nodos (tile_size={tile_size})")
    
    # 3. Construir la matriz
    grid = np.zeros((rows, cols), dtype=int)
    start = None
    goals = []
    
    for r in range(rows):
        for c in range(cols):
            # Extraer los píxeles de este tile
            r0, r1 = r * tile_size, (r + 1) * tile_size
            c0, c1 = c * tile_size, (c + 1) * tile_size
            tile = img_array[r0:r1, c0:c1]  # shape: (tile_size, tile_size, 3)
            
            # Clasificar el tile
            cell_type = classify_tile(tile)
            grid[r, c] = cell_type
            
            # Registrar inicio y metas
            if cell_type == START:
                start = (r, c)
            elif cell_type == GOAL:
                goals.append((r, c))
    
    print(f"Inicio encontrado en: {start}")
    print(f"Metas encontradas en: {goals}")
    
    return grid, start, goals, img_array


def visualize_grid(grid: np.ndarray, title: str = "Grilla Discreta"):
    # Mapa de colores: 0=blanco, 1=negro, 2=rojo, 3=verde
    color_map = {
        FREE:  [1.0, 1.0, 1.0],   # Blanco
        WALL:  [0.0, 0.0, 0.0],   # Negro
        START: [1.0, 0.0, 0.0],   # Rojo
        GOAL:  [0.0, 0.8, 0.0],   # Verde
    }
    
    rows, cols = grid.shape
    img_vis = np.ones((rows, cols, 3))
    for cell_type, color in color_map.items():
        mask = grid == cell_type
        img_vis[mask] = color
    
    plt.figure(figsize=(8, 8))
    plt.imshow(img_vis, interpolation='nearest')
    plt.title(title)
    plt.axis('off')
    
    # Leyenda
    patches = [
        mpatches.Patch(color='white', label='Libre', edgecolor='gray'),
        mpatches.Patch(color='black', label='Pared'),
        mpatches.Patch(color='red',   label='Inicio'),
        mpatches.Patch(color='green', label='Meta'),
    ]
    plt.legend(handles=patches, loc='upper right')
    plt.tight_layout()
    plt.show()

## Task 1.2 – Framework OOP y Búsqueda (BFS / DFS)

### Framework OOP

In [5]:
# clase base Problem que define qué debe poder hacer cualquier problema de búsqueda
# MazeProblem implementa ese contrato para nuestro laberinto.

class Problem(ABC): # Clase abstracta que representa cualquier problema de busqueda
    
    def __init__(self, initial_state): # Donde empieza el agente
        self.initial_state = initial_state
    
    @abstractmethod
    def actions(self, state): #Returns list[str] de acciones válidas desde 'state'
        pass
    
    @abstractmethod
    def result(self, state, action): # Returns el nuevo estado tras aplicar 'action' to 'state'
        pass
    
    @abstractmethod
    def goal_test(self, state) -> bool: # Returns True si 'state' es un estado meta
        pass
    
    def step_cost(self, state, action, next_state) -> float: # Costo de moverse
        return 1.0 # default 1


class MazeProblem(Problem): # Implementa Problem para un laberinto en grilla 2D, el estado es una tupla (fila, columna)
    
    # Las 4 acciones posibles: (delta_fila, delta_col)
    DIRECTIONS = {
        'UP':    (-1,  0),
        'DOWN':  ( 1,  0),
        'LEFT':  ( 0, -1),
        'RIGHT': ( 0,  1),
    }
    
    def __init__(self, grid: np.ndarray, start: tuple, goals: list):
        super().__init__(initial_state=start)
        self.grid  = grid
        self.goals = set(goals)  # Set para búsqueda O(1)
        self.rows, self.cols = grid.shape
    
    def actions(self, state: tuple) -> list:
        r, c = state
        valid = []
        for action, (dr, dc) in self.DIRECTIONS.items():
            nr, nc = r + dr, c + dc
            # Verificar límites del mapa sino no cuenta la acción
            if 0 <= nr < self.rows and 0 <= nc < self.cols:
                # Verificar que no sea pared
                if self.grid[nr, nc] != WALL:
                    valid.append(action)
        return valid
    
    def result(self, state: tuple, action: str) -> tuple: # Aplica la acción y retorna el nuevo estado (posición).
        r, c = state
        dr, dc = self.DIRECTIONS[action]
        return (r + dr, c + dc)
    
    def goal_test(self, state: tuple) -> bool: # Returns True si el estado es una de las metas.
        return state in self.goals
    
    def step_cost(self, state, action, next_state) -> float:
        return 1.0

### Busqueda

In [6]:
class Node:
    
    def __init__(self, state, parent=None, action=None, g_cost=0.0):
        self.state   = state # estado actual (posición en el laberinto) (x, y)
        self.parent  = parent # nodo padre (para reconstruir el camino)
        self.action  = action # acción que llevó a este nodo
        self.g_cost  = g_cost # costo acumulado desde el inicio hasta aquí
    
    def path(self) -> list: # reconstruye el camino desde el nodo inicial hasta este nodo
        states = []
        node = self
        while node is not None:
            states.append(node.state)
            node = node.parent
        states.reverse()  # De inicio a fin
        return states
    
    # Necesario para que el heap de A* pueda comparar nodos
    def __lt__(self, other):
        return self.g_cost < other.g_cost

In [7]:
# La lógica central que todos los algoritmos comparten.
# La diferencia entre BFS, DFS y A* está en la frontera.

def graph_search(problem: Problem, frontier): # problema y frontera

    # Nodo inicial
    start_node = Node(state=problem.initial_state)
    frontier.push(start_node)
    
    # Conjunto de estados ya visitados (evita ciclos)
    explored = set()
    
    while not frontier.is_empty():
        node = frontier.pop()
        
        # Verificar si llegamos a la meta
        if problem.goal_test(node.state):
            return node.path()
        
        # Si ya visitamos este estado, lo saltamos
        if node.state in explored:
            continue
        explored.add(node.state)
        
        # Expandir: generar hijos
        for action in problem.actions(node.state):
            next_state = problem.result(node.state, action)
            if next_state not in explored:
                cost = node.g_cost + problem.step_cost(node.state, action, next_state)
                child = Node(state=next_state, parent=node, action=action, g_cost=cost)
                frontier.push(child)
    
    return None  # No se encontró solución
    # Return lista de estados (el camino)


# Fronteras
class FIFOQueue: # implementa BFS, garantiza explorar por niveles (Camino mas corto)

    def __init__(self):
        self.queue = deque()
    
    def push(self, node):
        self.queue.append(node)   # Agrega al final
    
    def pop(self):
        return self.queue.popleft()  # Saca del frente
    
    def is_empty(self):
        return len(self.queue) == 0


class LIFOStack: # implementa DFS, garantiza explorar un camino profundo antes de retroceder

    def __init__(self):
        self.stack = []
    
    def push(self, node):
        self.stack.append(node)   # Agrega al tope
    
    def pop(self):
        return self.stack.pop()   # Saca del tope
    
    def is_empty(self):
        return len(self.stack) == 0

#Busquedas
def bfs(problem: Problem): #camino mas corto
    print("Ejecutando BFS...")
    path = graph_search(problem, FIFOQueue())
    if path:
        print(f"BFS encontró camino con {len(path)-1} pasos.")
    else:
        print("BFS: No se encontró solución.")
    return path


def dfs(problem: Problem): #explora rápido, no garantiza camino óptimo
    print("Ejecutando DFS...")
    path = graph_search(problem, LIFOStack())
    if path:
        print(f"DFS encontró camino con {len(path)-1} pasos.")
    else:
        print("DFS: No se encontró solución.")
    return path